# 1 — Importer les bibliothèques

In [1]:
import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

from xgboost import XGBClassifier

import joblib

In [2]:
print(mlflow.__version__)

3.18.0


# 2 — Configurer MLflow

In [3]:
# Connexion au serveur local
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Nom du projet MLflow
mlflow.set_experiment("fraud_detection")

print("Tracking URI :", mlflow.get_tracking_uri())

2026/11/06 10:35:32 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection' does not exist. Creating a new experiment.


Tracking URI : http://127.0.0.1:5000


# 3 - Charger les données et le preprocessing

In [5]:
root = Path.cwd()
print(root)

if root.name == "notebooks":
    root = root.parent

train = pd.read_parquet(root / "data/processed/train.parquet")
val = pd.read_parquet(root / "data/processed/validation.parquet")

X_train = train.drop(columns="Class")
y_train = train["Class"]

X_val = val.drop(columns="Class")
y_val = val["Class"]

preprocessor = joblib.load(
    root / "models/preprocessor.joblib"
)

# Calcul de la pondération
n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()

if n_positive == 0:
    raise ValueError("Aucune fraude dans le jeu d'entraînement.")

scale_pos_weight = n_negative / n_positive

print("scale_pos_weight :", scale_pos_weight)

C:\Users\benja\OneDrive\Bureau\test_jupyter\projet_MLops\notebooks
scale_pos_weight : 518.1770833333334


# 4 - Première expérience : régression logistique

In [7]:
with mlflow.start_run(run_name="Logistic_Regression"):

    model = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    # Entraînement
    model.fit(X_train, y_train)

    # Probabilités sur validation
    probabilities = model.predict_proba(X_val)[:, 1]

    # Seuil fixé à 0.5
    predictions = (probabilities >= 0.5).astype(int)

    # Calcul des métriques
    precision = precision_score(
        y_val, predictions, zero_division=0
    )

    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    ap = average_precision_score(
        y_val, probabilities
    )

    # Enregistrement des paramètres
    mlflow.log_params({
        "algorithm": "LogisticRegression",
        "max_iter": 1000,
        "class_weight": "balanced",
        "threshold": 0.5
    })

    # Enregistrement des métriques
    mlflow.log_metrics({
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "average_precision": ap
    })

    # Sauvegarde du pipeline complet
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        skops_trusted_types=["numpy.dtype"]
    )

    print("Precision :", precision)
    print("Recall :", recall)
    print("F1 :", f1)
    print("Average Precision :", ap)

Precision : 0.052845528455284556
Recall : 0.9285714285714286
F1 : 0.1
Average Precision : 0.8401374920229974
🏃 View run Logistic_Regression at: http://127.0.0.1:5000/#/experiments/1/runs/3545479729aa42efaba8e321ca222fd6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


# 6 - Deuxième expérience : XGBoost

In [11]:
with mlflow.start_run(run_name="XGBoost_v1"):

    xgb_model = Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            random_state=42
        ))
    ])

    xgb_model.fit(X_train, y_train)

    probabilities = xgb_model.predict_proba(X_val)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    precision = precision_score(
        y_val, predictions, zero_division=0
    )

    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    f1 = f1_score(
        y_val, predictions, zero_division=0
    )

    ap = average_precision_score(
        y_val, probabilities
    )

    mlflow.log_params({
        "algorithm": "XGBoost",
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "scale_pos_weight": float(scale_pos_weight),
        "threshold": 0.5
    })

    mlflow.log_metrics({
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "average_precision": ap
    })

    model_info = mlflow.sklearn.log_model(
        sk_model=xgb_model,
        name="model",
        skops_trusted_types= ['xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier', 'numpy.dtype']
    )

    print("Precision :", precision)
    print("Recall :", recall)
    print("F1 :", f1)
    print("Average Precision :", ap)

    print("Model URI :", model_info.model_uri)

Precision : 0.6428571428571429
Recall : 0.8035714285714286
F1 : 0.7142857142857143
Average Precision : 0.821748069101684
Model URI : models:/m-bfccf485539d48fb8fc6c8d46b72af63
🏃 View run XGBoost_v1 at: http://127.0.0.1:5000/#/experiments/1/runs/16d72176e9bc4959a965c3b2786a658e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


# 7 - Faire varier les hyperparamètres de XGBoost

C'est l'un des grands intérêts de MLflow : conserver les résultats de plusieurs configurations.

Nous allons comparer différentes profondeurs d'arbres.

In [13]:
for depth in [3, 4, 6, 8]:

    with mlflow.start_run(
        run_name=f"XGBoost_depth_{depth}"
    ):

        model = Pipeline([
            ("preprocessing", clone(preprocessor)),
            ("model", XGBClassifier(
                n_estimators=200,
                max_depth=depth,
                learning_rate=0.05,
                scale_pos_weight=scale_pos_weight,
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_val)[:, 1]

        predictions = (
            probabilities >= 0.5
        ).astype(int)

        ap = average_precision_score(
            y_val, probabilities
        )

        f1 = f1_score(
            y_val, predictions, zero_division=0
        )

        mlflow.log_params({
            "algorithm": "XGBoost",
            "max_depth": depth,
            "n_estimators": 200,
            "learning_rate": 0.05,
            "scale_pos_weight": float(scale_pos_weight),
            "threshold": 0.5
        })

        mlflow.log_metrics({
            "average_precision": ap,
            "f1": f1
        })

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            skops_trusted_types= ['xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier', 'numpy.dtype']
        )

        print(
            f"Depth={depth} | AP={ap:.4f} | F1={f1:.4f}"
        )

Depth=3 | AP=0.8497 | F1=0.2991
🏃 View run XGBoost_depth_3 at: http://127.0.0.1:5000/#/experiments/1/runs/abf15ef483c445aca3992a971261359b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Depth=4 | AP=0.8153 | F1=0.5806
🏃 View run XGBoost_depth_4 at: http://127.0.0.1:5000/#/experiments/1/runs/b27088a440e04512aef451e3e64a9902
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Depth=6 | AP=0.8126 | F1=0.8113
🏃 View run XGBoost_depth_6 at: http://127.0.0.1:5000/#/experiments/1/runs/1c9ec5e1d6d841908fc56a3bef704829
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Depth=8 | AP=0.8243 | F1=0.8077
🏃 View run XGBoost_depth_8 at: http://127.0.0.1:5000/#/experiments/1/runs/c8ff0ec7741545dba8f7b733340d5215
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


# 8 - Enregistrer le meilleur modèle dans le Model Registry
Le Model Registry permet de conserver plusieurs versions d'un modèle et d'identifier celle destinée à la production.

Nous allons sélectionner le run ayant obtenu la meilleure Average Precision parmi les expériences XGBoost.

In [16]:
import mlflow

# 1. Connexion au serveur MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# 2. Récupérer l'expérience
experiment = mlflow.get_experiment_by_name(
    "fraud_detection"
)

# Vérifier qu'elle existe
if experiment is None:
    raise ValueError("Expérience fraud_detection introuvable")

# 3. Récupérer les runs
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

# 4. Conserver uniquement les runs XGBoost
xgb_runs = runs[
    runs["params.algorithm"] == "XGBoost"
].copy()

# 5. Supprimer les runs sans métrique AP
xgb_runs = xgb_runs.dropna(
    subset=["metrics.average_precision"]
)

if xgb_runs.empty:
    raise ValueError("Aucun run XGBoost avec Average Precision")

# 6. Sélectionner le meilleur modèle
best_run = xgb_runs.sort_values(
    "metrics.average_precision",
    ascending=False
).iloc[0]

# 7. Récupérer son identifiant
best_run_id = best_run["run_id"]

print("Meilleur run :", best_run_id)
print(
    "Average Precision :",
    best_run["metrics.average_precision"]
)

Meilleur run : abf15ef483c445aca3992a971261359b
Average Precision : 0.8497200806876671


### Enregistrer model

In [17]:
model_uri = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="fraud_detection_model"
)

print("Nom :", registered_model.name)
print("Version :", registered_model.version)

Successfully registered model 'fraud_detection_model'.
2026/11/06 11:27:16 WARNING mlflow.tracking._model_registry.fluent: Run with id abf15ef483c445aca3992a971261359b has no artifacts at artifact path 'model', registering model based on models:/m-4d0a9364151b4db5aae216084c2e2bfb instead
2026/11/06 11:27:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud_detection_model, version 1


Nom : fraud_detection_model
Version : 1


Created version '1' of model 'fraud_detection_model'.


# 9 - Charger model

In [19]:
import mlflow.pyfunc

model_uri = (
    f"models:/fraud_detection_model/"
    f"{registered_model.version}"
)

loaded_model = mlflow.pyfunc.load_model(model_uri)

# Prédictions sur cinq transactions
sample = X_val.iloc[:5]

predictions = loaded_model.predict(sample)

print(predictions)

[0 0 0 0 0]
